<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.4-ai-chat-with-fastapi/notebooks/exercises-5_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.4: Building AI Chat with FastAPI

*Module 5 · Production Applications with GenAI APIs*

> LLMs have no memory. Every API call starts from zero. Your job: manage conversation history, fit it in the context window, persist it across sessions, and make the user feel like the AI "remembers" everything.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-5.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Chat Session Manager — Track Who Said What — `part1_session_manager.py`
2. Step 2 — System Prompt Patterns — Give Your Bot a Soul — `part2_system_prompts.py`
3. Step 3 — Context Window Strategies — When History Gets Too Long — `part3_memory_strategies.py`
4. Step 4 — Persistent Storage — Survive Server Restarts — `part4_persistent_sessions.py`
5. Step 5 — Project: Production Chat API — `project_chat_api.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai fastapi uvicorn redis pydantic


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Chat Session Manager — Track Who Said What

Each user gets their own conversation. Messages stay in order. Sessions are isolated.

**`part1_session_manager.py`** · _Block 1 of 5_

CHAT SESSION MANAGER — Each user gets their own conversation history.


In [ ]:
# =============================================
# CHAT SESSION MANAGER
# Each user gets their own conversation history.
# Messages are stored in the format the LLM expects.
# =============================================

import uuid
from datetime import datetime
from dataclasses import dataclass, field

@dataclass
class Message:
    role: str           # "user", "assistant", or "system"
    content: str
    timestamp: float = field(default_factory=lambda: datetime.now().timestamp())
    tokens: int = 0

class ChatSession:
    """One conversation between one user and the AI."""
    
    def __init__(self, session_id: str = None, system_prompt: str = ""):
        self.session_id = session_id or str(uuid.uuid4())[:8]
        self.messages: list[Message] = []
        self.created_at = datetime.now()
        
        if system_prompt:
            self.messages.append(Message(role="system", content=system_prompt))
    
    def add_user_message(self, content: str):
        self.messages.append(Message(role="user", content=content))
    
    def add_assistant_message(self, content: str):
        self.messages.append(Message(role="assistant", content=content))
    
    def to_api_format(self) -> list[dict]:
        """Convert to the format LLM APIs expect."""
        return [{"role": m.role, "content": m.content} for m in self.messages]
    
    def turn_count(self) -> int:
        return sum(1 for m in self.messages if m.role == "user")

class SessionStore:
    """Manage multiple chat sessions (one per user/conversation)."""
    
    def __init__(self):
        self.sessions: dict[str, ChatSession] = {}
    
    def create(self, system_prompt: str = "") -> ChatSession:
        session = ChatSession(system_prompt=system_prompt)
        self.sessions[session.session_id] = session
        return session
    
    def get(self, session_id: str) -> ChatSession | None:
        return self.sessions.get(session_id)
    
    def delete(self, session_id: str):
        self.sessions.pop(session_id, None)

# ── Demo ──
store = SessionStore()
session = store.create(system_prompt="You are a helpful AI tutor for the Netsetos course.")

session.add_user_message("What is RAG?")
session.add_assistant_message("RAG stands for Retrieval-Augmented Generation...")
session.add_user_message("How does the retrieval part work?")

print(f"Session: {session.session_id}")
print(f"Turns: {session.turn_count()}")
print(f"\nAPI format (what the LLM sees):")
for msg in session.to_api_format():
    print(f"  [{msg['role']:9s}] {msg['content'][:60]}...")


> **What just happened?** We built the foundation: a Message dataclass (role + content + timestamp), a ChatSession that tracks one conversation with to_api_format() for sending to any LLM, and a SessionStore that manages multiple sessions by ID. Every chat app starts here — before context management, before persistence, before anything else.


### Step 2 · System Prompt Patterns — Give Your Bot a Soul

The system prompt defines WHO the AI is, HOW it behaves, and WHAT it can/can't do.

**`part2_system_prompts.py`** · _Block 2 of 5_

SYSTEM PROMPT PATTERNS for different use cases. — Copy, customize, and use in your projects.


In [ ]:
# =============================================
# SYSTEM PROMPT PATTERNS for different use cases.
# Copy, customize, and use in your projects.
# =============================================

class SystemPromptBuilder:
    """Build structured system prompts from components."""
    
    def __init__(self):
        self.sections = {}
    
    def set_role(self, role: str):
        self.sections["role"] = f"# ROLE\n{role}"
        return self
    
    def set_rules(self, rules: list[str]):
        numbered = "\n".join([f"{i+1}. {r}" for i, r in enumerate(rules)])
        self.sections["rules"] = f"# RULES (never violate)\n{numbered}"
        return self
    
    def set_style(self, style: str):
        self.sections["style"] = f"# RESPONSE STYLE\n{style}"
        return self
    
    def set_knowledge(self, knowledge: str):
        self.sections["knowledge"] = f"# KNOWLEDGE BASE\n{knowledge}"
        return self
    
    def set_examples(self, examples: list[tuple[str, str]]):
        ex_text = "\n".join([f"User: {q}\nAssistant: {a}" for q, a in examples])
        self.sections["examples"] = f"# EXAMPLE INTERACTIONS\n{ex_text}"
        return self
    
    def build(self) -> str:
        return "\n\n".join(self.sections.values())

# ── Pattern 1: Customer Support Bot ──
support_prompt = (SystemPromptBuilder()
    .set_role("You are Asha, a customer support agent for Netsetos EdTech platform.")
    .set_rules([
        "Only answer questions about Netsetos courses, pricing, and technical issues.",
        "For billing issues, collect the order ID and escalate to human support.",
        "Never share internal policies, discount codes, or employee information.",
        "If asked about competitors, say 'I can only help with Netsetos questions.'",
        "Always be polite, even if the user is frustrated.",
    ])
    .set_style("Respond in 2-3 sentences. Use a warm, professional tone. Use the student's name if known.")
    .set_examples([
        ("How much is the GenAI course?", "The GenAI course is ₹29,999. It includes 14 modules, 146 hours of content, and 6 months of access. Would you like me to help with enrollment?"),
        ("Your course is too expensive", "I understand budget is important! We offer EMI options starting at ₹2,500/month. We also have early-bird discounts — want me to check if any are active?"),
    ])
    .build()
)

# ── Pattern 2: Coding Tutor ──
tutor_prompt = (SystemPromptBuilder()
    .set_role("You are a Python coding tutor for beginners. You teach by showing small code examples first, then explaining.")
    .set_rules([
        "Always show code FIRST, then explain what it does.",
        "Keep code examples under 15 lines.",
        "Use Indian analogies when helpful (cricket, chai, Bollywood).",
        "If the student makes a mistake, don't give the answer directly — ask guiding questions.",
        "End every response with a small exercise for the student to try.",
    ])
    .set_style("Friendly and encouraging. Use 'beta'/'beti' casually. Keep explanations under 100 words.")
    .build()
)

# ── Pattern 3: Dynamic System Prompt (inject real-time data) ──
def build_dynamic_prompt(user_name: str, user_plan: str, current_module: str) -> str:
    """System prompt that changes based on user context."""
    return (SystemPromptBuilder()
        .set_role(f"You are a learning assistant for {user_name}, currently on the {user_plan} plan.")
        .set_knowledge(f"""
Current progress: {current_module}
Plan: {user_plan} ({"full access" if user_plan == "pro" else "limited to Modules 1-3"})
Tip: If they ask about advanced topics beyond their plan, mention the upgrade option.""")
        .set_style("Personalized, encouraging. Reference their progress.")
        .build()
    )

print("Support bot prompt:")
print(support_prompt[:300] + "...\n")

print("Dynamic prompt for a specific user:")
print(build_dynamic_prompt("Rahul", "free", "Module 3: Prompt Engineering"))


> **What just happened?** We built a SystemPromptBuilder with chainable methods for: role, rules, style, knowledge base, and few-shot examples. Three patterns: (1) Customer support — strict rules about what to discuss, with example interactions. (2) Coding tutor — code-first teaching style with Indian analogies. (3) Dynamic prompts — system prompt changes based on user data (name, plan, progress). The dynamic pattern is how SaaS products personalize AI responses per user.


### Step 3 · Context Window Strategies — When History Gets Too Long

After 50 turns, your history is 30K tokens. It won't fit. Here's how to handle it.

**`part3_memory_strategies.py`** · _Block 3 of 5_

CONVERSATION MEMORY MANAGER — 3 strategies for keeping history within budget.


In [ ]:
# =============================================
# CONVERSATION MEMORY MANAGER
# 3 strategies for keeping history within budget.
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class MemoryManager:
    """Manage conversation memory within token budgets."""
    
    def __init__(self, max_history_tokens: int = 8000):
        self.max_tokens = max_history_tokens
        self.model = genai.GenerativeModel("gemini-2.0-flash")
    
    def _count(self, messages: list[dict]) -> int:
        text = "\n".join(m["content"] for m in messages)
        return self.model.count_tokens(text).total_tokens
    
    def needs_trimming(self, messages: list[dict]) -> bool:
        return self._count(messages) > self.max_tokens
    
    # ── Strategy 1: Sliding Window ──
    def sliding_window(self, messages: list[dict]) -> list[dict]:
        """Keep system prompt + last N messages that fit."""
        system = [m for m in messages if m["role"] == "system"]
        history = [m for m in messages if m["role"] != "system"]
        
        # Keep adding from most recent until budget is reached
        kept = []
        for msg in reversed(history):
            test = system + [msg] + kept
            if self._count(test) > self.max_tokens:
                break
            kept.insert(0, msg)
        
        return system + kept
    
    # ── Strategy 2: Summarize + Keep Recent ──
    def summarize_and_keep(self, messages: list[dict], keep_recent: int = 6) -> list[dict]:
        """Summarize old messages, keep recent ones in full."""
        system = [m for m in messages if m["role"] == "system"]
        history = [m for m in messages if m["role"] != "system"]
        
        if len(history) <= keep_recent:
            return messages
        
        old = history[:-keep_recent]
        recent = history[-keep_recent:]
        
        # Summarize old conversation
        old_text = "\n".join([f"{m['role']}: {m['content']}" for m in old])
        summary_resp = self.model.generate_content(
            f"Summarize this conversation in 3-4 bullet points. Keep key decisions, facts, and user preferences:\n\n{old_text}"
        )
        
        summary_msg = {"role": "system", "content": f"[Earlier conversation summary]\n{summary_resp.text}"}
        
        return system + [summary_msg] + recent
    
    # ── Strategy 3: Entity Memory (remember key facts) ──
    def extract_entities(self, messages: list[dict]) -> str:
        """Extract and maintain key facts about the user."""
        text = "\n".join([f"{m['role']}: {m['content']}" for m in messages if m["role"] != "system"])
        
        resp = self.model.generate_content(f"""Extract key facts from this conversation. Return as bullet points:
- User's name (if mentioned)
- User's goal/need
- Decisions made
- Preferences expressed
- Important numbers/dates mentioned

Conversation:
{text}

Key facts:""")
        return resp.text.strip()
    
    def trim(self, messages: list[dict], strategy: str = "summarize") -> list[dict]:
        """Trim messages using the specified strategy."""
        if not self.needs_trimming(messages):
            return messages
        
        if strategy == "window":
            return self.sliding_window(messages)
        elif strategy == "summarize":
            return self.summarize_and_keep(messages)
        return messages

# ── Demo ──
memory = MemoryManager(max_history_tokens=2000)

# Simulate a long conversation
long_chat = [
    {"role": "system", "content": "You are a helpful tutor."},
]
topics = ["RAG basics", "embeddings", "chunking", "vector DB", "retrieval",
          "reranking", "generation", "evaluation", "deployment", "monitoring"]

for t in topics:
    long_chat.append({"role": "user", "content": f"Explain {t} in detail with code examples."})
    long_chat.append({"role": "assistant", "content": f"{t} involves several important concepts..." * 10})

print(f"Original: {len(long_chat)} messages, {memory._count(long_chat)} tokens")

trimmed = memory.trim(long_chat, strategy="summarize")
print(f"Trimmed:  {len(trimmed)} messages, {memory._count(trimmed)} tokens")


> **What just happened?** A MemoryManager with 3 strategies: (1) Sliding window — drops oldest messages until it fits (fast, loses early context). (2) Summarize + keep recent — AI summarizes old messages into bullet points, keeps last 6 verbatim (best balance). (3) Entity memory — extracts key facts (name, goals, decisions) into a persistent profile. Strategy 2 is the production standard — users keep their recent conversation in full detail while earlier context is preserved as a concise summary.


### Step 4 · Persistent Storage — Survive Server Restarts

In-memory sessions vanish when the server restarts. Store them in Redis or a database.

**`part4_persistent_sessions.py`** · _Block 4 of 5_

PERSISTENT SESSIONS: Store in Redis — Sessions survive server restarts.


In [ ]:
# =============================================
# PERSISTENT SESSIONS: Store in Redis
# Sessions survive server restarts.
# pip install redis
# =============================================

import redis
import json
from datetime import datetime

class PersistentSessionStore:
    """Chat sessions stored in Redis — survive restarts."""
    
    def __init__(self, redis_url: str = "redis://localhost:6379", ttl_hours: int = 24):
        self.redis = redis.from_url(redis_url)
        self.ttl = ttl_hours * 3600
    
    def _key(self, session_id: str) -> str:
        return f"chat:session:{session_id}"
    
    def save(self, session: ChatSession):
        """Save a session to Redis."""
        data = {
            "session_id": session.session_id,
            "messages": [{"role": m.role, "content": m.content, "ts": m.timestamp} for m in session.messages],
            "created": session.created_at.isoformat(),
            "updated": datetime.now().isoformat(),
        }
        self.redis.setex(self._key(session.session_id), self.ttl, json.dumps(data))
    
    def load(self, session_id: str) -> ChatSession | None:
        """Load a session from Redis."""
        raw = self.redis.get(self._key(session_id))
        if not raw:
            return None
        
        data = json.loads(raw)
        session = ChatSession(session_id=data["session_id"])
        
        for m in data["messages"]:
            session.messages.append(Message(role=m["role"], content=m["content"], timestamp=m.get("ts", 0)))
        
        return session
    
    def list_sessions(self) -> list[str]:
        """List all active session IDs."""
        keys = self.redis.keys("chat:session:*")
        return [k.decode().split(":")[-1] for k in keys]

print("""
Why Redis for chat sessions?
  ✅ Fast: <1ms read/write (vs 5-10ms for database)
  ✅ TTL: sessions auto-expire after 24 hours  
  ✅ Scalable: handles 100K+ concurrent sessions
  ✅ Persistent: survives server restarts (with AOF/RDB)
  
For permanent history (analytics, compliance):
  → Also write to PostgreSQL or MongoDB asynchronously
  → Redis = fast session access, DB = permanent archive
""")


> **What just happened?** Production FastAPI layout follows the "domain module" pattern (16.5K-star fastapi-best-practices repo). Each feature is self-contained. Dependencies are cached within a request — calling get_db() twice in one request returns the same session. LLM clients must be lifespan singletons , not per-request — creating new HTTP connections wastes resources. Pydantic v2 is 5-50× faster than v1 thanks to its Rust core. Annotated type aliases make endpoints clean: async def chat(db: DbSession, user: CurrentUser, llm: LLMClient) .


### Step 5 · Project: Production Chat API

Sessions + memory + streaming + persistence — all in one deployable FastAPI app.

**`project_chat_api.py`** · _Block 5 of 5_

PRODUCTION CHAT API — Sessions + Memory + Streaming + Persistence


In [ ]:
# =============================================
# PRODUCTION CHAT API
# Sessions + Memory + Streaming + Persistence
# pip install fastapi uvicorn sse-starlette redis
# =============================================

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from sse_starlette.sse import EventSourceResponse
from pydantic import BaseModel
import google.generativeai as genai
import json, os, asyncio

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

app = FastAPI(title="Netsetos Chat API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# Global stores
sessions = SessionStore()
memory = MemoryManager(max_history_tokens=8000)
# persistent = PersistentSessionStore()  # uncomment with Redis running

DEFAULT_SYSTEM = (SystemPromptBuilder()
    .set_role("You are a helpful AI tutor for the Netsetos GenAI course.")
    .set_rules(["Keep answers under 150 words.", "Use code examples when helpful.", "Be encouraging."])
    .set_style("Friendly, concise, and practical. Use Indian context when relevant.")
    .build()
)

# ── Request/Response models ──
class CreateSession(BaseModel):
    system_prompt: str = DEFAULT_SYSTEM

class ChatMessage(BaseModel):
    message: str

class SessionInfo(BaseModel):
    session_id: str
    turns: int
    messages: list[dict]

# ── Endpoints ──

@app.post("/sessions", response_model=SessionInfo)
def create_session(req: CreateSession):
    """Create a new chat session."""
    session = sessions.create(system_prompt=req.system_prompt)
    return SessionInfo(session_id=session.session_id, turns=0, messages=[])

@app.get("/sessions/{session_id}", response_model=SessionInfo)
def get_session(session_id: str):
    """Get session history."""
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    return SessionInfo(
        session_id=session.session_id,
        turns=session.turn_count(),
        messages=session.to_api_format(),
    )

@app.post("/sessions/{session_id}/chat")
async def chat(session_id: str, req: ChatMessage):
    """Send a message and get a non-streaming response."""
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    
    # Add user message
    session.add_user_message(req.message)
    
    # Trim history if needed
    api_messages = memory.trim(session.to_api_format(), strategy="summarize")
    
    # Call LLM
    model = genai.GenerativeModel("gemini-2.0-flash")
    # Build content from messages
    system_msg = next((m["content"] for m in api_messages if m["role"] == "system"), "")
    chat_model = genai.GenerativeModel("gemini-2.0-flash", system_instruction=system_msg)
    
    # Build Gemini chat from history
    gemini_chat = chat_model.start_chat(history=[
        {"role": "user" if m["role"] == "user" else "model", "parts": [m["content"]]}
        for m in api_messages if m["role"] in ("user", "assistant")
    ][:-1])  # exclude last user msg — we'll send it as new
    
    response = gemini_chat.send_message(req.message)
    answer = response.text
    
    # Save assistant response
    session.add_assistant_message(answer)
    
    return {"response": answer, "turn": session.turn_count()}

@app.post("/sessions/{session_id}/chat/stream")
async def chat_stream(session_id: str, req: ChatMessage):
    """Send a message and get a streaming SSE response."""
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    
    session.add_user_message(req.message)
    api_messages = memory.trim(session.to_api_format(), strategy="summarize")
    
    async def generate():
        system_msg = next((m["content"] for m in api_messages if m["role"] == "system"), "")
        model = genai.GenerativeModel("gemini-2.0-flash", system_instruction=system_msg)
        
        stream = model.generate_content(req.message, stream=True)
        full = ""
        
        for chunk in stream:
            if chunk.text:
                full += chunk.text
                yield json.dumps({"type": "token", "content": chunk.text})
            await asyncio.sleep(0)
        
        session.add_assistant_message(full)
        yield json.dumps({"type": "done", "turn": session.turn_count()})
    
    return EventSourceResponse(generate())

@app.delete("/sessions/{session_id}")
def delete_session(session_id: str):
    sessions.delete(session_id)
    return {"deleted": session_id}

# Run: uvicorn project_chat_api:app --reload

print("""
Production Chat API Endpoints:

  POST   /sessions                        → Create new session
  GET    /sessions/{id}                   → Get session history
  POST   /sessions/{id}/chat              → Send message (non-streaming)
  POST   /sessions/{id}/chat/stream       → Send message (SSE streaming)
  DELETE /sessions/{id}                   → Delete session

Usage flow:
  1. POST /sessions → get session_id
  2. POST /sessions/{id}/chat/stream → send messages, get streaming responses
  3. GET  /sessions/{id} → retrieve full history
  4. DELETE /sessions/{id} → cleanup
  
Every turn:
  - User message added to history
  - History trimmed if over token budget (summarize strategy)
  - System prompt always preserved
  - Response streamed via SSE
  - Assistant response saved to history
""")


> **What just happened?** We built a complete production chat API: (1) Session management — create, get, delete sessions via REST endpoints. (2) Memory management — history is automatically trimmed with the summarize strategy when it exceeds 8K tokens. (3) Two response modes — non-streaming POST for simple integrations, SSE streaming for typed-out UX. (4) System prompt — built with SystemPromptBuilder, always preserved during trimming. (5) Persistent storage — Redis integration ready (uncomment one line). This is the backend architecture behind every AI chat product.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
